# CST 4702 Lab 8: Multi-Layer Perceptron

This notebook completes **Option 1: The Quadrant Challenge** from the lab handout.
I first train a standard Perceptron, then train an `MLPClassifier` with one hidden
layer and compare the two models on the same dataset.

The main idea is that this dataset behaves like an XOR pattern: opposite quadrants
belong to the same class, so a single straight line is not enough to separate them.


In [ ]:
from pathlib import Path
import site
import sys


user_site = Path(site.getusersitepackages())
if user_site.exists() and str(user_site) not in sys.path:
    sys.path.insert(0, str(user_site))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Perceptron
from sklearn.neural_network import MLPClassifier


In [ ]:
rng = np.random.default_rng(42)

# The assignment asks for 400 random 2D points with coordinates in [-2, 2].
X = rng.uniform(-2, 2, size=(400, 2))

# This label rule makes the problem XOR-like:
# top-right and bottom-left are +1, while the other two quadrants are -1.
y = np.where(X[:, 0] * X[:, 1] > 0, 1, -1)

# I use a train/test split so the accuracy reflects generalization,
# not just how well the model memorized the training data.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

def plot_decision_boundary(model, features, labels, title):
    # A dense mesh makes the learned shape easier to see.
    x_min, x_max = features[:, 0].min() - 0.25, features[:, 0].max() + 0.25
    y_min, y_max = features[:, 1].min() - 0.25, features[:, 1].max() + 0.25
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 400),
        np.linspace(y_min, y_max, 400),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = model.predict(grid).reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, zz, levels=[-1.5, 0, 1.5], alpha=0.28, cmap="coolwarm")
    plt.contour(xx, yy, zz, levels=[0], colors="black", linewidths=1.5)
    plt.scatter(
        features[:, 0],
        features[:, 1],
        c=labels,
        cmap="coolwarm",
        edgecolor="black",
        s=35,
    )
    plt.title(title)
    plt.xlabel("x")
    plt.ylabel("y")
    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)
    plt.show()

print(f"Dataset shape: {X.shape}")
print(f"Training points: {len(X_train)}")
print(f"Test points: {len(X_test)}")
print(f"Class balance: {-1: 195, 1: 205}")


Dataset shape: (400, 2)
Training points: 300
Test points: 100


ValueError: Invalid format specifier ' 195, 1: 205' for object of type 'int'

In [ ]:
perceptron = Perceptron(max_iter=1000, random_state=42)
perceptron.fit(X_train, y_train)

perceptron_accuracy = perceptron.score(X_test, y_test)
print(f"Perceptron test accuracy: {perceptron_accuracy:.3f}")


In [ ]:
plot_decision_boundary(
    perceptron,
    X,
    y,
    "Perceptron Decision Boundary on the Quadrant Dataset",
)


The Perceptron only reached **0.550** test accuracy, which is
exactly what we expect on this kind of dataset. A single-layer perceptron learns a
boundary of the form

$$
\hat{y} = \mathrm{sign}(w_1x + w_2y + b),
$$

so it can only split the plane with **one straight line**. Here the true labeling
rule depends on the **sign of the product** $xy$, which means the positive class
lives in the top-right and bottom-left quadrants while the negative class lives in
the other two quadrants. No single line can isolate those two diagonal regions at
the same time.


In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(15,),
    activation="relu",
    random_state=42,
    max_iter=1000,
)
mlp.fit(X_train, y_train)

mlp_accuracy = mlp.score(X_test, y_test)
print(f"MLP test accuracy: {mlp_accuracy:.3f}")


In [ ]:
plot_decision_boundary(
    mlp,
    X,
    y,
    "MLP Decision Boundary on the Quadrant Dataset",
)


In [ ]:
perceptron_predictions = perceptron.predict(X_test)
mlp_predictions = mlp.predict(X_test)

# I purposely look for a point the perceptron misses and the MLP fixes,
# because it makes the comparison much easier to talk through.
candidate_indices = np.where(
    (perceptron_predictions != y_test) & (mlp_predictions == y_test)
)[0]
chosen_index = int(candidate_indices[0]) if len(candidate_indices) else 0

test_point = X_test[chosen_index]
actual_label = int(y_test[chosen_index])
perceptron_label = int(perceptron_predictions[chosen_index])
mlp_label = int(mlp_predictions[chosen_index])

print(f"Chosen test point: {test_point}")
print(f"Actual label: {actual_label}")
print(f"Perceptron prediction: {perceptron_label}")
print(f"MLP prediction: {mlp_label}")
print(f"Point product x*y: {test_point[0] * test_point[1]:.4f}")


On the same split, the MLP reached **0.960** test accuracy.

For the selected test point **(0.260, 0.290)**, the
product is **0.0753**, so the correct class is **1**.
The Perceptron predicted **-1**, while the MLP predicted
**1**. Perceptron misclassified this point while the MLP got it right.

Mathematically, the hidden layer changes the model from one linear score to a
composition of linear pieces:

$$
h = \mathrm{ReLU}(W_1x + b_1), \qquad
\hat{y} = \mathrm{sign}(W_2h + b_2).
$$

Each hidden neuron can respond to a different region of the plane, and the output
layer can combine those responses into a **non-linear** decision surface. That extra
flexibility is what lets the MLP approximate the quadrant rule, while the plain
Perceptron is stuck with a single straight boundary.
